Engineering LLM Systems - Tokenization & State Management

In [4]:
%pip install tiktoken

     -------------------------------------- 944.4/944.4 kB 2.1 MB/s eta 0:00:00
     -------------------------------------- 278.3/278.3 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: c:\Users\pruch\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


Section 1: Tokenization of Structured Data

In [5]:
import tiktoken

# Load Encoing model
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode a String
tokens = encoding.encode("Hello, world! Hello, world! Hello, world! Hello, world! Hello, world!")
print(f"Total tokens: {len(tokens)}")

Total tokens: 20


Section 2: Implementing State in Stateless Architectures

In [6]:
from json import load
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
print("Environment initialized for Architectural Review demo.")


Environment initialized for Architectural Review demo.


Step 1: Defining the System Context

In [7]:
system_description = """
Our architecture uses a Node.js API Gateway, three Python microservices for data processing, 
and a PostgreSQL primary database with two read-replicas. We use RabbitMQ for asynchronous 
event processing between the services.
"""

messages = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": f"Please acknowledge this system architecture: {system_description}"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Your system architecture appears to be well-structured and leverages various technologies effectively. Here's a brief acknowledgment of each component:

1. **Node.js API Gateway**: Using Node.js for the API Gateway allows for non-blocking I/O, which is excellent for handling a high number of requests efficiently. This can serve as a single entry point for your API consumers, enabling you to manage authentication, rate limiting, and route requests to the appropriate microservices.

2. **Python Microservices**: Python is a great choice for data processing tasks due to its rich ecosystem of libraries and frameworks. By implementing three microservices, you can encapsulate different functionalities or business logic, promoting separation of concerns and making the architecture more scalable.

3. **PostgreSQL Database with Read Replicas**: PostgreSQL is a robust relational database choice, and having a primary database with two read replicas can significantly enhance read perf

Step 2: The Stateless Failure

In [8]:
# Attempting to ask about the replicas without restating the architecture
messages = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": "How should we handle the failover for the read-replicas?"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Handling failover for read replicas is an important aspect of designing a resilient architecture, particularly in cloud environments. Here's a structured approach to manage failover for read replicas effectively:

### 1. **Monitoring and Detection**

- **Health Checks**: Implement regular health checks on read replicas to monitor their status. Use cloud-native monitoring tools (like AWS CloudWatch for AWS or Azure Monitor for Azure) to track metrics such as latency, error rates, and resource utilization.
- **Automated Alerts**: Set up alerts to notify your team in case of failures or threshold breaches.

### 2. **Automated Failover Mechanism**

- **Database Services with Built-in Failover**: Use managed database services that provide automatic failover capabilities (e.g., AWS RDS with Multi-AZ, Azure SQL Database with auto-failover groups).
  
- **Custom Failover Logic**: If using self-managed replicas or custom databases, implement logic to promote a standby replica to a

The Engineering Solution: Message Threading

To solve this, we must maintain a history array. In a production environment, this would likely be stored in a Redis cache or a database indexed by a SessionID.

In [9]:
history = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": f"Architecture: {system_description}"},
    {"role": "assistant", "content": "I have reviewed the Node.js/Python/PostgreSQL stack with RabbitMQ. How can I help?"},
    {"role": "user", "content": "How should we handle the failover for the read-replicas?"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=history)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Handling failover for read-replicas in a PostgreSQL setup is crucial to ensure high availability and maintain performance levels. Here’s how you can implement failover for your PostgreSQL read-replicas:

### 1. Monitoring and Health Checks
Regularly monitor the health and performance of your read-replicas. Use monitoring tools and set up health checks to detect if a replica goes down. Popular tools include:

- **PgHero**: Provides insights about your Postgres databases and their health.
- **Prometheus/Grafana**: For custom metrics and alerts.
- **AWS CloudWatch** (if using AWS RDS): Monitors database metrics and can trigger alerts.

### 2. Automated Failover Mechanism
Implement a mechanism to automatically failover to a standby read-replica when the primary read-replica fails:

- **Patroni**: A template for PostgreSQL high availability, useful if you want to implement automatic failover.
- **pg_auto_failover**: A tool designed to manage failover automatically in PostgreSQ